<a href="https://colab.research.google.com/github/annatsamoyra-prog/data-story-/blob/main/tovima_texnologia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import logging
import random
import re
import time
from datetime import datetime
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry



In [ ]:
BASE_URL = "https://www.tovima.gr/category/texnologia/"
MAX_PAGES = 6
TARGET_YEAR = 2026
MIN_DELAY = 1.5   # δευτερόλεπτα ανάμεσα σε requests (ευγενικό scraping)
MAX_DELAY = 3.0
REQUEST_TIMEOUT = 15
OUTPUT_CSV = "tovima_texnologia_2026.csv"
OUTPUT_JSON = "tovima_texnologia_2026.json"


In [ ]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    ),
    "Accept-Language": "el-GR,el;q=0.9,en-US;q=0.8,en;q=0.7",
}

GREEK_MONTHS = {
    "ιανουαρίου": 1, "φεβρουαρίου": 2, "μαρτίου": 3, "απριλίου": 4,
    "μαΐου": 5, "ιουνίου": 6, "ιουλίου": 7, "αυγούστου": 8,
    "σεπτεμβρίου": 9, "οκτωβρίου": 10, "νοεμβρίου": 11, "δεκεμβρίου": 12,
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("tovima_scraper")


In [ ]:
def build_session() -> requests.Session:
    session = requests.Session()
    session.headers.update(HEADERS)
    retries = Retry(
        total=3,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    session.mount("https://", HTTPAdapter(max_retries=retries))
    session.mount("http://", HTTPAdapter(max_retries=retries))
    return session


In [ ]:
def parse_greek_date(date_text: str):
    """Προσπαθεί να μετατρέψει κείμενο ημερομηνίας σε datetime.
    Επιστρέφει None αν αποτύχει — τότε γίνεται fallback σε string check."""
    if not date_text:
        return None

    # Try numeric format: DD.MM.YYYY
    numeric_match = re.search(
        r"(\d{1,2})\.(\d{1,2})\.(\d{4})",
        date_text.strip(),
    )
    if numeric_match:
        day, month, year = numeric_match.groups()
        try:
            return datetime(int(year), int(month), int(day))
        except ValueError:
            pass # Fall through to Greek month parsing if numeric parse fails

    # Fallback to Greek month name format: DD MonthName YYYY
    match = re.search(
        r"(\d{1,2})\s+([Α-Ωα-ωάέήίόύώΐΰ]+)\s+(\d{4})",
        date_text.strip(),
        re.IGNORECASE,
    )
    if match:
        day, month_name, year = match.groups()
        month = GREEK_MONTHS.get(month_name.lower())
        if month:
            try:
                return datetime(int(year), month, int(day))
            except ValueError:
                pass # Return None below if this also fails

    return None

In [ ]:
def extract_article(story: BeautifulSoup, page: int, base_url: str):
    # Find the date tag and text
    date_tag = story.find("span", class_="date-time")
    # Extract text directly from the span tag
    date_text = date_tag.get_text(strip=True) if date_tag else None
    parsed_date = parse_greek_date(date_text)

    # If the article date is not for the TARGET_YEAR, skip it.
    if parsed_date and parsed_date.year != TARGET_YEAR:
        return None
    # If parsed_date is None, and date_text exists, we can still do a string check
    elif not parsed_date and date_text and str(TARGET_YEAR) not in date_text:
        return None


    headline_tag = story.find("h3")
    headline = headline_tag.get_text(" ", strip=True) if headline_tag else None

    # Ο σύνδεσμος του άρθρου: προτίμηση στο <a> μέσα στον τίτλο,
    # αλλιώς fallback στο πρώτο <a> του block.
    link_tag = None
    if headline_tag is not None:
        link_tag = headline_tag.find("a", href=True)
    if link_tag is None:
        link_tag = story.find("a", href=True)

    article_url = urljoin(base_url, link_tag["href"]) if link_tag else None

    summary_tag = story.find("p")
    summary = summary_tag.get_text(" ", strip=True) if summary_tag else None

    author_tag = story.find("span", class_="vima-author")
    author = author_tag.get_text(" ", strip=True) if author_tag else None

    return {
        "headline": headline,
        "date": date_text,
        "parsed_date": parsed_date.isoformat() if parsed_date else None,
        "author": author,
        "summary": summary,
        "url": article_url,
        "page": page,
    }


# --------------------------------------------------
# Κύρια λούπα scraping
# --------------------------------------------------

def scrape() -> pd.DataFrame:
    session = build_session()
    articles_list = []

    for page in range(1, MAX_PAGES + 1):
        url = BASE_URL if page == 1 else f"{BASE_URL}page/{page}/"
        log.info("Σελίδα %d: %s", page, url)

        try:
            response = session.get(url, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
        except requests.RequestException as exc:
            log.warning("Αποτυχία στη σελίδα %d (%s) — προσπερνάω.", page, exc)
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        stories = soup.find_all("div", class_="wrap-category-row")
        log.info("Βρέθηκαν %d article blocks", len(stories))

        if not stories:
            log.info("Καμία ανάρτηση στη σελίδα %d — σταματάω το pagination.", page)
            break

        for story in stories:
            try:
                article = extract_article(story, page, url)
            except Exception as exc:  # δεν θέλουμε ένα κακό block να ρίξει όλο το scraping
                log.warning("Σφάλμα στην εξαγωγή άρθρου: %s", exc)
                continue
            if article is not None:
                articles_list.append(article)

        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

    df = pd.DataFrame(articles_list)
    if not df.empty:
        df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)
    return df


def save_outputs(df: pd.DataFrame):
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    df.to_json(OUTPUT_JSON, orient="records", force_ascii=False, indent=2)
    log.info("Αποθηκεύτηκε: %s (%d γραμμές)", OUTPUT_CSV, len(df))
    log.info("Αποθηκεύτηκε: %s", OUTPUT_JSON)


if __name__ == "__main__":
    result_df = scrape()
    log.info("Συνολικά μοναδικά άρθρα του %d: %d", TARGET_YEAR, len(result_df))
    if not result_df.empty:
        print(result_df.head(10).to_string())
        save_outputs(result_df)
    else:
        log.warning("Δεν βρέθηκαν άρθρα — έλεγξε τα selectors (class names) του site.")

                                                                                                   headline  date parsed_date                 author                                                                                                                                                                                                                                                           summary                                                                                                                                        url  page
0                                   Grand Theft Auto VI: Η ημέρα, η ώρα και η πλατφόρμα της παρουσίασης του  None        None               Newsroom                                                                                                                                      Πολλές λεπτομέρειες για το Grand Theft Auto VI αναμένεται να γίνουν γνωστές στην παρουσίαση του παιχνιδιού από την Rockstar.                         https://www.tovim

In [ ]:
result_df

,headline,date,parsed_date,author,summary,url,page
0,"Grand Theft Auto VI: Η ημέρα, η ώρα και η πλατ...",None,None,Newsroom,Πολλές λεπτομέρειες για το Grand Theft Auto VI...,https://www.tovima.gr/2026/08/24/texnologia/gr...,1
1,"Οι νεκροί επιστρέφουν, αλλά όχι όπως τους θυμό...",None,None,Κωνσταντίνος Δέδες,"Ο θάνατος, η απόλυτη ανθρώπινη εμπειρία, περνά...",https://www.tovima.gr/2026/08/23/texnologia/me...,1
2,Τεχνητή νοημοσύνη: Γιατί η γενιά Z και οι νέοι...,None,None,Newsroom,Πάνω από το 50% των Αμερικανών εκφράζει φόβο γ...,https://www.tovima.gr/2026/08/20/texnologia/te...,1
3,Meta Glasses: Μήπως σε καταγράφουν και δεν το ...,None,None,Κατερίνα Μανιαδάκη,"Μια κάμερα που τη φοράς, ένα μικρό φωτάκι που ...",https://www.tovima.gr/2026/08/20/texnologia/me...,1
4,ChatGPT: «Φρένο» στους εφήβους βάζει η OpenAI ...,None,None,Newsroom,Τι αλλάζει στο ChatGPT για τους ανήλικους χρήσ...,https://www.tovima.gr/2026/08/18/texnologia/ch...,1
...,...,...,...,...,...,...,...
107,Γιατί η χρήση AI για απλές εργασίες δεν είναι ...,None,None,Κωνσταντίνος Δέδες,"Αν όλες οι απλές εργασίες αυτοματοποιηθούν, ο ...",https://www.tovima.gr/2026/01/12/texnologia/gi...,6
108,Οι συσκεύες και τα gadgets του μέλλοντος: όσα ...,None,None,Ντιάνα Καρτσαγκούλη,"Τεχνολογίες που κάνουν τη ζωή μας πιο εύκολή, ...",https://www.tovima.gr/2026/01/10/texnologia/ce...,6
109,Η OpenAI «μπαίνει» στην Υγεία: Η νέα λειτουργί...,None,None,None,Η OpenAI ανοίγει ένα νέο κεφάλαιο στην ψηφιακή...,https://www.tovima.gr/2026/01/09/texnologia/i-...,6
110,Grok: Στο στόχαστρο των Αρχών το Χ του Έλον Μα...,None,None,None,Το εργαλείο τεχνητής νοημοσύνης Grok χρησιμοπο...,https://www.tovima.gr/2026/01/08/texnologia/gr...,6


In [ ]:
df_tovima = scrape()
df_tovima

,headline,date,parsed_date,author,summary,url,page
0,"Grand Theft Auto VI: Η ημέρα, η ώρα και η πλατ...",None,None,Newsroom,Πολλές λεπτομέρειες για το Grand Theft Auto VI...,https://www.tovima.gr/2026/08/24/texnologia/gr...,1
1,"Οι νεκροί επιστρέφουν, αλλά όχι όπως τους θυμό...",None,None,Κωνσταντίνος Δέδες,"Ο θάνατος, η απόλυτη ανθρώπινη εμπειρία, περνά...",https://www.tovima.gr/2026/08/23/texnologia/me...,1
2,Τεχνητή νοημοσύνη: Γιατί η γενιά Z και οι νέοι...,None,None,Newsroom,Πάνω από το 50% των Αμερικανών εκφράζει φόβο γ...,https://www.tovima.gr/2026/08/20/texnologia/te...,1
3,Meta Glasses: Μήπως σε καταγράφουν και δεν το ...,None,None,Κατερίνα Μανιαδάκη,"Μια κάμερα που τη φοράς, ένα μικρό φωτάκι που ...",https://www.tovima.gr/2026/08/20/texnologia/me...,1
4,ChatGPT: «Φρένο» στους εφήβους βάζει η OpenAI ...,None,None,Newsroom,Τι αλλάζει στο ChatGPT για τους ανήλικους χρήσ...,https://www.tovima.gr/2026/08/18/texnologia/ch...,1
...,...,...,...,...,...,...,...
107,Γιατί η χρήση AI για απλές εργασίες δεν είναι ...,None,None,Κωνσταντίνος Δέδες,"Αν όλες οι απλές εργασίες αυτοματοποιηθούν, ο ...",https://www.tovima.gr/2026/01/12/texnologia/gi...,6
108,Οι συσκεύες και τα gadgets του μέλλοντος: όσα ...,None,None,Ντιάνα Καρτσαγκούλη,"Τεχνολογίες που κάνουν τη ζωή μας πιο εύκολή, ...",https://www.tovima.gr/2026/01/10/texnologia/ce...,6
109,Η OpenAI «μπαίνει» στην Υγεία: Η νέα λειτουργί...,None,None,None,Η OpenAI ανοίγει ένα νέο κεφάλαιο στην ψηφιακή...,https://www.tovima.gr/2026/01/09/texnologia/i-...,6
110,Grok: Στο στόχαστρο των Αρχών το Χ του Έλον Μα...,None,None,None,Το εργαλείο τεχνητής νοημοσύνης Grok χρησιμοπο...,https://www.tovima.gr/2026/01/08/texnologia/gr...,6


In [ ]:
import base64
import requests
from google.colab import userdata

def save_df_to_github(df, repo, path, token=None, branch="main", message="Update dataset"):
    token = token or userdata.get("newtoken")
    csv_content = df.to_csv(index=False, encoding="utf-8-sig")
    content_b64 = base64.b64encode(csv_content.encode("utf-8-sig")).decode("utf-8")

    url = f"https://api.github.com/repos/{repo}/contents/{path}"
    headers = {"Authorization": f"token {token}", "Accept": "application/vnd.github+json"}

    existing = requests.get(url, headers=headers, params={"ref": branch})
    sha = existing.json().get("sha") if existing.status_code == 200 else None

    payload = {"message": message, "content": content_b64, "branch": branch}
    if sha:
        payload["sha"] = sha

    response = requests.put(url, headers=headers, json=payload)
    response.raise_for_status()
    print(f"Αποθηκεύτηκε: https://github.com/{repo}/blob/{branch}/{path}")
    return response.json()

In [ ]:
save_df_to_github(
    df_tovima,
    repo="annatsamoyra-prog/data-story-",
    path="tovima_texnologia_2026.csv"
)

Αποθηκεύτηκε: https://github.com/annatsamoyra-prog/data-story-/blob/main/tovima_texnologia_2026.csv


{'content': {'name': 'tovima_texnologia_2026.csv',
  'path': 'tovima_texnologia_2026.csv',
  'sha': 'ab3a2c33c10af1964115278dd0db3933b5f4c362',
  'size': 68583,
  'url': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/contents/tovima_texnologia_2026.csv?ref=main',
  'html_url': 'https://github.com/annatsamoyra-prog/data-story-/blob/main/tovima_texnologia_2026.csv',
  'git_url': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/git/blobs/ab3a2c33c10af1964115278dd0db3933b5f4c362',
  'download_url': 'https://raw.githubusercontent.com/annatsamoyra-prog/data-story-/main/tovima_texnologia_2026.csv',
  'type': 'file',
  '_links': {'self': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/contents/tovima_texnologia_2026.csv?ref=main',
   'git': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/git/blobs/ab3a2c33c10af1964115278dd0db3933b5f4c362',
   'html': 'https://github.com/annatsamoyra-prog/data-story-/blob/main/tovima_texnologia_2026.csv'}},
 'comm